In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append("../src")
from cv.data_prep import verify_video, extract_uniform_frames

RAW_VIDEOS_DIR = Path("../data/raw_videos_v2")
FRAMES_DIR = Path("../data/frames_v2")
PROCESSED_DIR = Path("../data/processed")

RAW_VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
all_videos = list(RAW_VIDEOS_DIR.glob("*.mp4"))
valid_data = []

for v_path in all_videos:
    meta = verify_video(v_path)
    if meta:
        valid_data.append({"video_id": v_path.stem, "path": str(v_path), **meta})
    else:
        print(f"Invalid or corrupt video: {v_path.name}")

df_valid = pd.DataFrame(valid_data)
df_valid.to_csv(PROCESSED_DIR / "valid_videos_v2.csv", index=False)
print(f"Found {len(df_valid)} valid videos.")

Found 50 valid videos.


In [3]:
all_extracted = []

for _, row in df_valid.iterrows():
    frames = extract_uniform_frames(row["path"], FRAMES_DIR, row["video_id"])
    all_extracted.extend(frames)

df_frames = pd.DataFrame(all_extracted)
print(f"Extracted {len(df_frames)} total frames.")

Extracted 400 total frames.


In [4]:
# Cell 4: Train / Validation / Test Split and Leakage Checks
import sys
import os
from pathlib import Path

root_dir = str(Path(os.getcwd()).resolve().parents[0]) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

import pandas as pd
from src.cv.validation import split_video_dataset, run_leakage_and_integrity_check

raw_vids_dir = os.path.join(root_dir, "data", "raw_videos_v2")
video_files = [f for f in os.listdir(raw_vids_dir) if f.endswith(".mp4")]
df_meta = pd.DataFrame({"video_id": [os.path.splitext(f)[0] for f in video_files]})

train_df, val_df, test_df = split_video_dataset(df_meta, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42)

processed_dir = os.path.join(root_dir, "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

train_path = os.path.join(processed_dir, "train_v2.csv")
val_path = os.path.join(processed_dir, "val_v2.csv")
test_path = os.path.join(processed_dir, "test_v2.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

is_passed = run_leakage_and_integrity_check(train_path, val_path, test_path)

print(f"AmirAli Shahabi: Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")
print(f"AmirAli Shahabi: Integrity check passed: {is_passed}")

AmirAli Shahabi: Train size: 34, Val size: 8, Test size: 8
AmirAli Shahabi: Integrity check passed: True


In [9]:
from pathlib import Path
import pandas as pd

root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent

processed_dir = root_dir / "data" / "processed"
frames_dir = root_dir / "data" / "frames_v2"

splits = {
    "train": pd.read_csv(processed_dir / "train_v2.csv"),
    "val": pd.read_csv(processed_dir / "val_v2.csv"),
    "test": pd.read_csv(processed_dir / "test_v2.csv"),
}

records = []

for frame_path in sorted(frames_dir.iterdir()):
    if not frame_path.is_file() or frame_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    video_id_raw = frame_path.name.split("_frame_")[0]
    # Normalize video id by converting to int and back to string to match formats like '20' vs '00020'
    try:
        video_id = str(int(video_id_raw))
    except ValueError:
        video_id = video_id_raw

    records.append({
        "frame_path": frame_path.relative_to(root_dir).as_posix(),
        "video_id": video_id,
        "video_id_raw": video_id_raw
    })

frames_df = pd.DataFrame(records)

if frames_df.empty:
    raise ValueError(f"No image frames found in {frames_dir}")

for split_name, videos_df in splits.items():
    # Normalize split video ids similarly
    split_video_ids = set()
    for v in videos_df["video_id"].dropna():
        try:
            split_video_ids.add(str(int(v)))
        except ValueError:
            split_video_ids.add(str(v))

    split_df = frames_df[frames_df["video_id"].isin(split_video_ids)].copy()
    split_df["label"] = ""

    output_path = processed_dir / f"{split_name}_frames_v2.csv"
    split_df[["frame_path", "video_id_raw", "label"]].rename(columns={"video_id_raw": "video_id"}).to_csv(output_path, index=False)

    print(f"{split_name}: {len(split_video_ids)} videos, {len(split_df)} frames")
    print(f"Saved: {output_path}")

train: 34 videos, 272 frames
Saved: C:\Users\Afrano\Documents\Quera\my_project\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer\data\processed\train_frames_v2.csv
val: 8 videos, 64 frames
Saved: C:\Users\Afrano\Documents\Quera\my_project\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer\data\processed\val_frames_v2.csv
test: 8 videos, 64 frames
Saved: C:\Users\Afrano\Documents\Quera\my_project\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer-20261029T081712Z-1-001\incident-sentiment-analyzer\data\processed\test_frames_v2.csv
